# yolonot local System One judge — fine-tune Laya on Colab

This notebook trains a local typed decision model ("System One") that answers the
exact questions yolonot's typed judge sends (`ml/spec/yolonot_systemone.json`,
generated from `internal/yolonot/decision.go`): one risk-tier choice and nine
yes/no ask conditions. Served over `/v1/systemone`, it replaces the chat LLM in
the judge chain with no Go changes: `fast_allow → yolonot-s1`.

Pipeline:

1. Generate synthetic commands with a teacher LLM and label them (tier + 9 conditions).
   Commands that match the shipped eval suites are dropped, so the suites stay an honest test set.
2. Fine-tune [Laya](https://github.com/NandhaKishorM/laya) (ModernBERT-large, 421M) starting from its
   `typed-decisions` checkpoint.
3. Serve the checkpoint and run `yolonot eval` on the greenfield and script-attach suites against
   the zero-shot base model (and Jev, if you have a key).
4. Save the checkpoint to Drive for the Mac.

**Runtime:** GPU. A T4 works with the smaller settings below; L4/A100 is much faster.
**Secrets** (Colab key icon): `GEMINI_API_KEY` for the teacher, optional `OPENROUTER_API_KEY`
to label with Jev and to compare against it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup

In [ ]:
# Parameters — edit these.
YOLONOT_REPO   = "https://github.com/sezaakgun/yolonot"
YOLONOT_BRANCH = "claude/project-thread-cz8cub"  # switch to "main" once ml/ is merged

TEACHER_URL    = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
TEACHER_MODEL  = "gemini-2.5-flash"  # any OpenAI-compatible chat model; a stronger one gives better labels
TEACHER_KEY    = "GEMINI_API_KEY"    # Colab secret name

LABEL_WITH_JEV = False               # True: label with Jev over OpenRouter (needs OPENROUTER_API_KEY)
JEV_URL, JEV_MODEL = "https://openrouter.ai/api/v1/systemone", "typesafe/jev-1.13"

TARGET_ROWS    = 3000                # 300 for a smoke run; 3000-6000 for a real one
BASE_MODEL     = "convaiinnovations/laya-typed-decisions"
EPOCHS         = 3
MICRO_BATCH    = 4                   # lower to 2 on a T4 if you hit OOM
GRAD_ACCUM     = 16
MAX_LEN        = 2048                # the default hint sections alone are ~800 tokens
HEAD_MAX_LEN   = 512                 # the question criteria are long

WORK = "/content/work"
SAVE_TO_DRIVE = True

In [ ]:
import os, subprocess
os.makedirs(WORK, exist_ok=True)
if not os.path.exists("/content/yolonot"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", YOLONOT_BRANCH, YOLONOT_REPO, "/content/yolonot"], check=True)
%pip install -q "laya[serve]" requests pandas

In [ ]:
from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return ""
os.environ[TEACHER_KEY] = secret(TEACHER_KEY)
os.environ["OPENROUTER_API_KEY"] = secret("OPENROUTER_API_KEY")
assert os.environ[TEACHER_KEY], f"add the {TEACHER_KEY} secret"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/yolonot-ml"
    os.makedirs(DRIVE_DIR, exist_ok=True)

## 2. Generate and label training data

Resumable: re-running appends until `TARGET_ROWS`. With Drive on, the file lives there so a
disconnected runtime loses nothing.

In [ ]:
DATA = (DRIVE_DIR if SAVE_TO_DRIVE else WORK) + "/train.jsonl"
cmd = ["python", "-m", "yolonot_ml.generate", "--out", DATA, "--target", str(TARGET_ROWS),
       "--gen-url", TEACHER_URL, "--gen-model", TEACHER_MODEL, "--gen-key-env", TEACHER_KEY, "--workers", "8"]
if LABEL_WITH_JEV:
    cmd += ["--label-protocol", "systemone", "--label-url", JEV_URL, "--label-model", JEV_MODEL,
            "--label-key-env", "OPENROUTER_API_KEY"]
subprocess.run(cmd, cwd="/content/yolonot/ml", check=True)

In [ ]:
import json, pandas as pd
rows = [json.loads(l) for l in open(DATA)]
meta = pd.DataFrame([r["meta"] for r in rows])
print(len(rows), "rows")
display(meta.teacher_risk.value_counts())
display(meta.teacher_decision.value_counts())
display(meta.groupby("scenario").teacher_risk.agg(lambda s: s.value_counts().to_dict()))
conds = [q for q, v in rows[0]["questions"].items() if v["type"] == "noul"]
display(pd.Series({c: sum(r["gold"][c]["probabilities"]["true"] >= 0.5 for r in rows) for c in conds}, name="fires"))

Spot-check a few labels before spending GPU time on them. Bad teacher labels are the main way this goes wrong.

In [ ]:
for r in meta.sample(15, random_state=0).itertuples():
    print(f"{r.teacher_risk:9} {r.teacher_decision:5} {r.scenario:18} {r.command[:110]}")

## 3. Fine-tune

In [ ]:
CKPT = WORK + "/yolonot-s1"
train = ["laya-train", "--data", DATA, "--base", BASE_MODEL, "--out", CKPT,
         "--epochs", str(EPOCHS), "--micro-batch", str(MICRO_BATCH), "--grad-accum", str(GRAD_ACCUM),
         "--max-len", str(MAX_LEN), "--head-max-len", str(HEAD_MAX_LEN), "--device", "cuda"]
# Dry run first: counts items, shows skipped questions and the optimizer-update budget.
subprocess.run(train + ["--dry-run"], check=True)

In [ ]:
subprocess.run(train, check=True)

## 4. Evaluate with yolonot's own eval suites

Both models are served by one `laya-serve` on loopback; yolonot's real typed-judge code
sends the requests, so the numbers are what the hook would do. Everything runs under a
throw-away `HOME`, never a real `~/.yolonot`.

In [ ]:
import time, json, urllib.request
srv_env = dict(os.environ, LAYA_HOST="127.0.0.1", LAYA_PORT="8009",
               LAYA_EXTRA_MODELS=json.dumps({"yolonot-s1": CKPT}),
               LAYA_MODELS="typed-decisions,yolonot-s1", LAYA_MAX_LOADED="2")
server = subprocess.Popen(["laya-serve"], env=srv_env, stdout=open(WORK + "/serve.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8009/health", timeout=2); break
    except Exception:
        time.sleep(2)
!tail -3 {WORK}/serve.log

In [ ]:
# Go toolchain + yolonot binary
GO = "1.26.2"
if not os.path.exists("/usr/local/go/bin/go"):
    !wget -q https://go.dev/dl/go{GO}.linux-amd64.tar.gz -O /tmp/go.tgz && tar -C /usr/local -xzf /tmp/go.tgz
!cd /content/yolonot && /usr/local/go/bin/go build -o /content/yolonot-bin .

In [ ]:
EVAL_HOME = WORK + "/eval-home"
os.makedirs(EVAL_HOME + "/.yolonot", exist_ok=True)
models = {
    "yolonot-s1": {"protocol": "systemone", "url": "http://127.0.0.1:8009/v1/systemone", "model": "yolonot-s1", "timeout": 30},
    "laya-base":  {"protocol": "systemone", "url": "http://127.0.0.1:8009/v1/systemone", "model": "typed-decisions", "timeout": 30},
}
if os.environ.get("OPENROUTER_API_KEY"):
    models["jev"] = {"protocol": "systemone", "url": JEV_URL, "model": JEV_MODEL, "env_key": "OPENROUTER_API_KEY"}
json.dump({"models": models, "judges": ["fast_allow", "yolonot-s1"]}, open(EVAL_HOME + "/.yolonot/config.json", "w"), indent=2)

def yolonot_eval(*args):
    env = dict(os.environ, HOME=EVAL_HOME)
    return subprocess.run(["/content/yolonot-bin", "eval", *args], cwd="/content/yolonot", env=env)

model_flags = sum((["--model", m] for m in models), [])
yolonot_eval("--suite", "evals/suites/greenfield.jsonl", *model_flags, "--runs", "1", "--output", WORK + "/greenfield.json")

In [ ]:
yolonot_eval("--suite", "evals/suites/greenfield.jsonl", *model_flags, "--metric", "risk", "--runs", "1")
yolonot_eval("--suite", "evals/suites/script-attach.jsonl", *model_flags, "--runs", "1")

What to look for: **dangerous catch must stay at 100%** (no `ask`/`deny` case allowed). After that,
higher decision accuracy (fewer needless asks) and latency. Jev scored 92.0% decision accuracy on
greenfield in the docs. If the fine-tune does not beat `laya-base` clearly, fix the data
(the label spot-check above) before training longer.

## 5. Save the checkpoint

In [ ]:
server.terminate()
if SAVE_TO_DRIVE:
    !cd {WORK} && tar czf {DRIVE_DIR}/yolonot-s1.tgz yolonot-s1 greenfield.json
    print("saved to", DRIVE_DIR + "/yolonot-s1.tgz")

On the Mac, see `ml/README.md` → *Run it locally*: unpack the archive, start `laya-serve` on
loopback, and add the model to yolonot's judge chain.